![clothing_classification](clothing_classification.png)


Fashion Forward is a new AI-based e-commerce clothing retailer.
They want to use image classification to automatically categorize new product listings, making it easier for customers to find what they're looking for. It will also assist in inventory management by quickly sorting items.

As a data scientist tasked with implementing a garment classifier, your primary objective is to develop a machine learning model capable of accurately categorizing images of clothing items into distinct garment types such as shirts, trousers, shoes, etc.

In [64]:
# Run the cells below first

In [65]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchmetrics import Accuracy, Precision, Recall

In [66]:
# Load datasets
from torchvision import datasets
import torchvision.transforms as transforms

train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

dataloader_train = DataLoader(train_data, shuffle=True, batch_size=32)
dataloader_test = DataLoader(test_data, batch_size=32)
    
'''image, label = next(iter(dataloader_train))
image = image.squeeze()
print(image.shape)
print("Class names:", train_data.classes)'''


class Net(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Flatten(), 
        )
        self.classifier = nn.Linear(64*7*7, num_classes)
    
    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.classifier(x)
        return x

net = Net(num_classes=10)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

for epoch in range(1):
    for images, label in dataloader_train:
        optimizer.zero_grad()
        outputs = net(images)
        loss = criterion(outputs, label)
        loss.backward()
        optimizer.step()       

metric_precision = Precision(
task="multiclass", num_classes=10, average="none")

metric_recall = Recall(
task="multiclass", num_classes=10, average="none")

metric_accuracy = Accuracy(
task="multiclass", num_classes=10)

net.eval()
predictions = []
with torch.no_grad():
    for images, labels in dataloader_test:
        outputs = net(images)
        _, preds = torch.max(outputs, 1)
        predictions.extend(preds.tolist())
        metric_precision(preds, labels)
        metric_recall(preds, labels)
        metric_accuracy(preds,labels)
precision = metric_precision.compute().tolist()
recall = metric_recall.compute().tolist()
accuracy = float(metric_accuracy.compute())

In [67]:
# Start coding here


# Use as many cells as you need